# LLM judge citation fidelity

**Question: does every cited passage in an Evidence Lab brief actually say what its cited sources say?**

This notebook is a fact-checker for one brief. It runs in three steps, each with its own output a reviewer can open:

| Step | What happens | Where to look |
|---|---|---|
| 1. Load | Reads a saved brief straight from the database and pulls out every passage that carries a `[n]` citation, together with the cited document, its page and section, and the **exact excerpt text the system stored when it wrote the passage**. Nothing is re-fetched or re-searched. | Tables below + `notebooks/output/citations_<brief-id>.xlsx` |
| 2. Judge | Sends each passage with its cited excerpts to an LLM judge that must decide whether the passage is supported *by those excerpts only*, quoting the supporting text verbatim. The prompts are printed in full, and every quote the judge returns is checked programmatically against the excerpt. | Prompt cells in section 3 |
| 3. Flag | Lists every passage the judge did not rate as fully supported, with its reasoning next to the passage and the source text, so a human can confirm or dismiss each one quickly. | Section 4 + `notebooks/output/judge_flagged_<brief-id>.xlsx` |

**How to run**

```bash
python3.11 -m venv ~/.venvs/evidencelab-notebooks
~/.venvs/evidencelab-notebooks/bin/pip install -r notebooks/requirements.txt
~/.venvs/evidencelab-notebooks/bin/jupyter lab notebooks/llm_judge_citation_fidelity.ipynb
```

Needs the local Docker stack's Postgres on `localhost:5432` (the `.env` credentials are reused) and `AZURE_FOUNDRY_KEY` / `AZURE_FOUNDRY_ENDPOINT` in `.env` for the judge. Run the cells top to bottom; the only settings you may want to change are marked `# <-- setting`.

In [ ]:
import asyncio
import hashlib
import html as html_mod
import json
import os
import re
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import psycopg2
from IPython.display import HTML, Markdown, display

# Repo root: works whether Jupyter was started from the repo root or notebooks/.
REPO_ROOT = Path.cwd() if (Path.cwd() / "notebooks").is_dir() else Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT / "notebooks"))

from citation_fidelity_lib import (  # noqa: E402
    FLAGGED_VERDICTS,
    build_judge_messages,
    extract_cited_passages,
    judgement_row,
    parse_judge_response,
    passage_source_rows,
    researched_sections,
    source_body,
    source_label,
    source_section,
    verify_quotes,
    write_review_workbook,
)

pd.set_option("display.max_colwidth", 70)
pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 60)


def read_env(path: Path) -> dict:
    """Minimal KEY=VALUE parser for the repo .env (no extra dependency)."""
    env = {}
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, value = line.partition("=")
        env[key.strip()] = value.strip().strip('"').strip("'")
    return env


DOTENV = read_env(REPO_ROOT / ".env")

# .env holds the Docker-network hostname (POSTGRES_HOST=postgres); from the host
# we connect to the published port on localhost. Override via environment.
PG = dict(
    host=os.environ.get("EVIDENCELAB_PG_HOST", "localhost"),
    port=int(os.environ.get("EVIDENCELAB_PG_PORT", "5432")),
    dbname=DOTENV.get("POSTGRES_DBNAME") or DOTENV.get("POSTGRES_DB", "evidencelab"),
    user=DOTENV["POSTGRES_USER"],
    password=DOTENV["POSTGRES_PASSWORD"],
)
conn = psycopg2.connect(**PG)

OUTPUT_DIR = REPO_ROOT / "notebooks" / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Connected to postgres://{PG['user']}@{PG['host']}:{PG['port']}/{PG['dbname']}")
print(f"Outputs go to {OUTPUT_DIR}")

## 1. Pick a brief

All saved briefs, most recently updated first. By default the notebook takes the most recent brief that has at least one cited passage; set `BRIEF_ID` to analyse another one.

In [ ]:
with conn.cursor() as cur:
    cur.execute(
        "SELECT id, title, data_source, updated_at, content FROM briefs ORDER BY updated_at DESC"
    )
    brief_rows = [
        {"id": str(r[0]), "title": r[1], "data_source": r[2], "updated_at": r[3], "content": r[4]}
        for r in cur
    ]


def count_cited_passages(content: dict) -> int:
    return sum(len(extract_cited_passages(s)) for s in researched_sections(content))


overview = pd.DataFrame(
    [
        {
            "id": r["id"],
            "title": r["title"],
            "data_source": r["data_source"],
            "updated": r["updated_at"].date(),
            "researched_sections": len(researched_sections(r["content"])),
            "cited_passages": count_cited_passages(r["content"]),
        }
        for r in brief_rows
    ]
)
display(overview)

BRIEF_ID = None  # <-- setting: a brief id from the table above, or None for the default

if BRIEF_ID is None:
    BRIEF_ID = next(r["id"] for r in brief_rows if count_cited_passages(r["content"]) > 0)
brief_row = next(r for r in brief_rows if r["id"] == BRIEF_ID)
brief = brief_row["content"]
brief_title = brief_row["title"]
sections = researched_sections(brief)
print(f"Analysing brief {BRIEF_ID}")
print(f"  title:        {brief_title}")
print(f"  data source:  {brief_row['data_source']}")
print(f"  researched:   {len(sections)} of {len(brief.get('sections') or [])} sections")

## 2. What the brief cites: passages and their source text

A **passage** is one sentence of the brief that carries citation markers such as `[3]` or `[1, 5]` (the same sentence split the UI uses for its citation highlights). Every number resolves to a **source** the system stored with the brief: the document title, page and section of the source document, and the exact excerpt text the research step read. That excerpt is what the judge sees later, so it is also what a reviewer should compare against.

In [ ]:
passages = []
for section in sections:
    passages.extend(extract_cited_passages(section))

section_overview = pd.DataFrame(
    [
        {
            "brief_section": section["title"],
            "cited_passages": sum(p.section_title == section["title"] for p in passages),
            "citations": sum(
                len(p.citation_indices) for p in passages if p.section_title == section["title"]
            ),
            "distinct_documents": len(
                {
                    src.get("docId")
                    for p in passages
                    if p.section_title == section["title"]
                    for src in p.sources
                }
            ),
            "dangling_citations": sum(
                len(p.missing_indices) for p in passages if p.section_title == section["title"]
            ),
        }
        for section in sections
    ]
)
display(section_overview)

dangling = [p for p in passages if p.dangling]
print(f"{len(passages)} cited passages across {len(sections)} sections.")
if dangling:
    print(
        f"WARNING: {len(dangling)} passage(s) cite a number with no stored source "
        "(dangling citation); they are listed in the Excel export and flagged in section 4."
    )

### A worked example

One passage rendered the way the rest of this notebook presents evidence: the brief passage first, then each cited source (click to expand the excerpt). This is exactly the material the judge receives for that passage.

In [ ]:
def _esc(text) -> str:
    return html_mod.escape(str(text if text is not None else ""))


def _highlight(body: str, spans) -> str:
    """HTML-escape the excerpt and wrap each located quote span in <mark>."""
    out = _esc(body)
    for span in spans or []:
        tokens = [re.escape(_esc(t)) for t in span.split()]
        if not tokens:
            continue
        pattern = re.compile(r"\s+".join(tokens))
        out = pattern.sub(lambda m: f"<mark>{m.group(0)}</mark>", out, count=1)
    return out


QUOTE_MARKS = {
    "verbatim": "✓",
    "near": "≈ near-verbatim (source text has PDF extraction artefacts)",
    "missing": "✗ NOT FOUND in the cited excerpt",
}


VERDICT_COLOURS = {
    "supported": "#2e7d32",
    "partially_supported": "#ef6c00",
    "unsupported": "#c62828",
    "cannot_assess": "#6d6d6d",
}


def render_passage_card(passage, pid=None, judgement=None, open_sources=False) -> str:
    """One passage with its cited excerpts and, if present, the judge's verdict."""
    head = f"<b>Passage{'' if pid is None else f' #{pid}'}</b> · {_esc(passage.section_title)}"
    parts = [f"<div style='margin:6px 0 4px'>{head}</div>"]
    parts.append(
        f"<div style='padding:8px 12px;background:#f5f5f5;border-radius:6px'>"
        f"<i>{_esc(passage.passage)}</i></div>"
    )
    quotes = (judgement or {}).get("supporting_quotes") or []
    checks = verify_quotes(passage, quotes) if judgement else []
    if judgement:
        colour = VERDICT_COLOURS.get(judgement["verdict"], "#000")
        parts.append(
            f"<div style='margin:8px 0'><b style='color:{colour}'>"
            f"{_esc(judgement['verdict'].upper())}</b> "
            f"(confidence {judgement.get('confidence', 0):.2f})<br>"
            f"{_esc(judgement.get('explanation', ''))}</div>"
        )
        if judgement.get("problems"):
            items = "".join(f"<li>{_esc(p)}</li>" for p in judgement["problems"])
            parts.append(f"<div><b>Problems:</b><ul style='margin:2px 0'>{items}</ul></div>")
        if quotes:
            items = "".join(
                f"<li>{QUOTE_MARKS[check['status']]} [{_esc(q.get('citation'))}] "
                f"“{_esc(q['quote'])}”</li>"
                for q, check in zip(quotes, checks)
            )
            parts.append(
                f"<div><b>Supporting quotes:</b><ul style='margin:2px 0'>{items}</ul></div>"
            )
    for source in passage.sources:
        own_spans = [
            span
            for q, check in zip(quotes, checks)
            if q.get("citation") in (source.get("index"), None)
            for span in check["spans"]
        ]
        section = source_section(source)
        summary = _esc(source_label(source)) + (f" — <i>{_esc(section)}</i>" if section else "")
        parts.append(
            f"<details{' open' if open_sources else ''} style='margin:4px 0 4px 12px'>"
            f"<summary>{summary}</summary>"
            f"<pre style='white-space:pre-wrap;font-size:12px;background:#fafafa;"
            f"padding:8px;border-radius:6px'>{_highlight(source_body(source), own_spans)}</pre>"
            f"</details>"
        )
    for index in passage.missing_indices:
        parts.append(
            f"<div style='margin:4px 0 4px 12px;color:#c62828'>[{index}] "
            "no source stored for this citation number</div>"
        )
    return (
        "<div style='border:1px solid #ddd;border-radius:8px;padding:8px 12px;margin:10px 0'>"
        + "".join(parts)
        + "</div>"
    )


EXAMPLE_PASSAGE_ID = None  # <-- setting: a passage_id, or None for the first multi-source passage

if EXAMPLE_PASSAGE_ID is None:
    EXAMPLE_PASSAGE_ID = next(
        (i for i, p in enumerate(passages, start=1) if len(p.sources) >= 2), 1
    )
example = passages[EXAMPLE_PASSAGE_ID - 1]
display(HTML(render_passage_card(example, EXAMPLE_PASSAGE_ID, open_sources=True)))

### Export: passage → source document → source section → excerpt

`citations_<brief-id>.xlsx` has one row per (passage, cited source) so each citation can be checked on its own line, plus a one-row-per-passage sheet and the per-section overview.

In [ ]:
citation_rows = passage_source_rows(passages)
citations_df = pd.DataFrame(citation_rows)

passage_rows = [
    {
        "passage_id": pid,
        "brief_section": p.section_title,
        "passage": p.passage_clean,
        "citations": ", ".join(str(i) for i in p.citation_indices),
        "sources": len(p.sources),
        "documents": " | ".join(f"[{s.get('index')}] {s.get('title')}" for s in p.sources),
        "dangling_citations": ", ".join(str(i) for i in p.missing_indices),
    }
    for pid, p in enumerate(passages, start=1)
]

CITATIONS_XLSX = OUTPUT_DIR / f"citations_{BRIEF_ID}.xlsx"
write_review_workbook(
    {
        "Passages": passage_rows,
        "Passage-Sources": citation_rows,
        "Sections": section_overview.to_dict("records"),
    },
    str(CITATIONS_XLSX),
)
print(f"Saved {len(passage_rows)} passages / {len(citation_rows)} citation rows to {CITATIONS_XLSX}")

preview = citations_df[
    ["passage_id", "brief_section", "passage", "citation", "document", "page", "source_section"]
].head(12)
display(preview)

## 3. The LLM judge

**Design**

- One judge call per passage. The judge sees the brief section title, the passage (citation markers removed) and the full excerpt of every source the passage cites, labelled with the citation number, document, page and section.
- The judge may use **only those excerpts**: no outside knowledge, no benefit of the doubt. It returns one of four verdicts, the supporting text quoted **verbatim**, and a list of concrete problems.
- Every quote it returns is checked programmatically against the excerpt it attributes it to (`quotes_verified` column): **verbatim** (same words, punctuation aside), **near-verbatim** (same words, but the stored text has PDF extraction artefacts such as "G overnment" that the judge repaired), or **not found**. A verdict resting on a quote that is not in the source is marked (`quote_not_in_source`), because that is the classic way a judge model hallucinates.
- Verdicts other than `supported` are flagged for human review. A flag is a lead, not a conviction: the judge only sees the stored excerpt, so a claim can be true elsewhere in the same document and still be flagged here.
- Model: `gpt-4.1-mini` on Azure Foundry, the same deployment the app's assistant uses. Change `JUDGE_MODEL` below to any other deployment name to get a second opinion.
- Responses are cached per (model, prompt) in `judge_cache_<brief-id>.jsonl`, so re-running the notebook does not re-spend tokens; delete the file to judge from scratch. The cache doubles as the audit log: it holds the exact messages sent and the raw answer.

**The prompts, exactly as sent**

In [ ]:
JUDGE_SYSTEM_PROMPT = """You are a meticulous fact-checker auditing an evidence brief written from evaluation reports.

You will receive ONE passage from the brief and the EXACT excerpts of the source documents that the passage cites. Decide whether the passage is supported by these excerpts and nothing else.

Rules:
1. Use ONLY the excerpts provided. Do not use outside knowledge, and do not assume the rest of the document says more.
2. Check every factual element of the passage: figures and units, dates and periods, countries, programmes and organisations, populations, the direction and size of effects, causal or comparative wording, and the strength of the claim (a source saying "may" or "in one district" does not support "does" or "nationally").
3. A passage that generalises across several sources is fine if each excerpt supports its share of the generalisation.
4. Paraphrase is fine; changed meaning is not.

Verdicts:
- "supported": every factual element is stated in, or directly follows from, the excerpts.
- "partially_supported": the core statement is in the excerpts, but at least one element is absent, overstated, or altered.
- "unsupported": the excerpts do not back the core statement, or they contradict it.
- "cannot_assess": the excerpts are unreadable or empty.

Answer with a single JSON object and nothing else:
{
  "verdict": "supported" | "partially_supported" | "unsupported" | "cannot_assess",
  "confidence": <number between 0 and 1>,
  "supporting_quotes": [{"citation": <citation number>, "quote": "<short VERBATIM quote copied from that excerpt>"}],
  "problems": ["<one concrete problem per entry: what the passage says vs. what the excerpt says; empty if supported>"],
  "explanation": "<one to three sentences for a human reviewer>"
}
Quotes must be copied character for character from the excerpt they are attributed to. Never invent or paraphrase a quote."""

JUDGE_USER_TEMPLATE = """Brief section: {brief_section}

Passage to check (it cites {citations}):
\"\"\"{passage}\"\"\"

Cited source excerpts:

{excerpts}"""

print(JUDGE_SYSTEM_PROMPT)

The rendered user message for the worked example above, so the judge's input is visible end to end:

In [ ]:
example_messages = build_judge_messages(example, JUDGE_SYSTEM_PROMPT, JUDGE_USER_TEMPLATE)
display(
    HTML(
        "<details open><summary><b>User message for passage #"
        f"{EXAMPLE_PASSAGE_ID}</b> ({len(example_messages[1]['content'])} characters)</summary>"
        f"<pre style='white-space:pre-wrap;font-size:12px'>{_esc(example_messages[1]['content'])}"
        "</pre></details>"
    )
)

### Run the judge

In [ ]:
from openai import AsyncOpenAI

JUDGE_MODEL = "gpt-4.1-mini"  # <-- setting: Azure Foundry deployment name
JUDGE_MAX_PASSAGES = None  # <-- setting: e.g. 20 for a quick look, None = all passages
JUDGE_CONCURRENCY = 6  # parallel requests
JUDGE_MAX_OUTPUT_TOKENS = 900

# Same URL scheme as utils/llm_factory.py::_create_azure_foundry_llm.
_endpoint = DOTENV["AZURE_FOUNDRY_ENDPOINT"].rstrip("/")
if "/openai/deployments" not in _endpoint:
    _endpoint = f"{_endpoint}/openai/deployments/{JUDGE_MODEL}"
judge_client = AsyncOpenAI(
    api_key=DOTENV["AZURE_FOUNDRY_KEY"],
    base_url=_endpoint,
    default_query={"api-version": DOTENV.get("AZURE_FOUNDRY_API_VERSION", "2024-02-15-preview")},
)

CACHE_PATH = OUTPUT_DIR / f"judge_cache_{BRIEF_ID}.jsonl"
cache = {}
if CACHE_PATH.exists():
    for line in CACHE_PATH.read_text().splitlines():
        record = json.loads(line)
        cache[record["key"]] = record
print(f"Judge: {JUDGE_MODEL} · {len(cache)} cached answers in {CACHE_PATH.name}")


def cache_key(messages) -> str:
    return hashlib.sha256(
        json.dumps({"model": JUDGE_MODEL, "messages": messages}, sort_keys=True).encode()
    ).hexdigest()


async def judge_passage(pid, passage, semaphore):
    """Return the cache record (messages + raw answer) for one passage."""
    messages = build_judge_messages(passage, JUDGE_SYSTEM_PROMPT, JUDGE_USER_TEMPLATE)
    key = cache_key(messages)
    if key in cache:
        return cache[key]
    async with semaphore:
        response = await judge_client.chat.completions.create(
            model=JUDGE_MODEL,
            messages=messages,
            temperature=0,
            response_format={"type": "json_object"},
            max_completion_tokens=JUDGE_MAX_OUTPUT_TOKENS,
        )
    record = {
        "key": key,
        "passage_id": pid,
        "model": JUDGE_MODEL,
        "messages": messages,
        "raw": response.choices[0].message.content,
        "usage": {
            "prompt_tokens": response.usage.prompt_tokens,
            "completion_tokens": response.usage.completion_tokens,
        },
        "judged_at": datetime.now(timezone.utc).isoformat(),
    }
    cache[key] = record
    with CACHE_PATH.open("a") as handle:
        handle.write(json.dumps(record, ensure_ascii=False) + "\n")
    return record


to_judge = [(pid, p) for pid, p in enumerate(passages, start=1) if p.sources]
if JUDGE_MAX_PASSAGES is not None:
    to_judge = to_judge[:JUDGE_MAX_PASSAGES]
skipped = [(pid, p) for pid, p in enumerate(passages, start=1) if not p.sources]

t0 = time.time()
semaphore = asyncio.Semaphore(JUDGE_CONCURRENCY)
records = await asyncio.gather(*(judge_passage(pid, p, semaphore) for pid, p in to_judge))
elapsed = time.time() - t0

judgements = {}
parse_errors = []
for (pid, passage), record in zip(to_judge, records):
    try:
        judgements[pid] = parse_judge_response(record["raw"])
    except ValueError as exc:
        parse_errors.append((pid, str(exc)))
        judgements[pid] = {
            "verdict": "cannot_assess",
            "confidence": 0.0,
            "supporting_quotes": [],
            "problems": [f"JUDGE ERROR: {exc}"],
            "explanation": "The judge's answer could not be parsed; see the raw answer in the cache file.",
        }
    if passage.missing_indices:
        judgements[pid]["problems"].append(
            "Dangling citation(s) "
            + ", ".join(f"[{i}]" for i in passage.missing_indices)
            + ": no source stored for these numbers."
        )
for pid, passage in skipped:
    judgements[pid] = {
        "verdict": "cannot_assess",
        "confidence": 0.0,
        "supporting_quotes": [],
        "problems": [
            "No source stored for citation(s) "
            + ", ".join(f"[{i}]" for i in passage.missing_indices)
        ],
        "explanation": "Not sent to the judge: nothing to check the passage against.",
    }

prompt_tokens = sum(r["usage"]["prompt_tokens"] for r in records)
completion_tokens = sum(r["usage"]["completion_tokens"] for r in records)
print(
    f"Judged {len(to_judge)} passages in {elapsed:.0f}s "
    f"({prompt_tokens:,} prompt + {completion_tokens:,} completion tokens incl. cached); "
    f"{len(skipped)} passage(s) skipped for lack of any stored source."
)
if parse_errors:
    print(f"WARNING: {len(parse_errors)} judge answer(s) could not be parsed: {parse_errors}")

## 4. Results

### Overview

In [ ]:
results_df = pd.DataFrame(
    [judgement_row(pid, passages[pid - 1], judgements[pid]) for pid in sorted(judgements)]
)

verdict_order = ["supported", "partially_supported", "unsupported", "cannot_assess"]
totals = results_df["verdict"].value_counts().reindex(verdict_order, fill_value=0)
by_section = (
    pd.crosstab(results_df["brief_section"], results_df["verdict"])
    .reindex(columns=verdict_order, fill_value=0)
)
by_section["flagged_share"] = (
    1 - by_section["supported"] / by_section[verdict_order].sum(axis=1)
).round(2)
display(by_section)

flagged_df = results_df[results_df["flagged"]].copy()
severity = {"unsupported": 0, "cannot_assess": 1, "partially_supported": 2}
flagged_df["severity"] = flagged_df["verdict"].map(severity)
flagged_df = flagged_df.sort_values(["severity", "confidence"], ascending=[True, False]).drop(
    columns="severity"
)

print(
    f"{len(results_df)} passages judged: "
    + ", ".join(f"{n} {v}" for v, n in totals.items())
    + f". Flagged for review: {len(flagged_df)}."
)
bad_quotes = int(results_df["quote_not_in_source"].sum())
if bad_quotes:
    print(
        f"NOTE: {bad_quotes} judgement(s) cite a supporting quote that could not be found in the "
        "cited excerpt; treat their verdicts with extra care (column quote_not_in_source)."
    )

### Flagged passages, at a glance

One line per flagged passage, most serious first. Passage and problems are cut short here; the cards below and the spreadsheet carry the full text.

In [ ]:
def _short(text, n=110) -> str:
    text = str(text or "").replace("\n", " · ")
    return text if len(text) <= n else text[: n - 1] + "…"


glance = flagged_df.assign(
    passage=flagged_df["passage"].map(_short),
    problems=flagged_df["problems"].map(_short),
)[["passage_id", "brief_section", "verdict", "confidence", "citations", "passage", "problems"]]
display(glance.reset_index(drop=True))

### Flagged passages, in detail

Each card shows the passage, the judge's verdict and reasoning, and the cited excerpts (collapsed; click to expand). The quotes the judge relied on are highlighted in the excerpt where they were found.

In [ ]:
MAX_CARDS = 30  # <-- setting: how many flagged passages to render here (all are in the Excel)

cards = [
    render_passage_card(passages[int(row.passage_id) - 1], int(row.passage_id), judgements[int(row.passage_id)])
    for row in flagged_df.head(MAX_CARDS).itertuples()
]
if len(flagged_df) > MAX_CARDS:
    cards.append(
        f"<p><i>{len(flagged_df) - MAX_CARDS} more flagged passage(s) in the spreadsheet.</i></p>"
    )
display(HTML("".join(cards) if cards else "<p>No passages flagged.</p>"))

### Save the review

`judge_flagged_<brief-id>.xlsx` holds the flagged passages first, then every judgement and the per-section overview. The full prompts and raw answers stay in `judge_cache_<brief-id>.jsonl`.

In [ ]:
FLAGGED_XLSX = OUTPUT_DIR / f"judge_flagged_{BRIEF_ID}.xlsx"
summary_rows = by_section.reset_index().to_dict("records")
summary_rows.append(
    {"brief_section": "TOTAL", **totals.to_dict(), "flagged_share": round(1 - totals["supported"] / max(totals.sum(), 1), 2)}
)
write_review_workbook(
    {
        "Flagged": flagged_df.to_dict("records"),
        "All judgements": results_df.to_dict("records"),
        "Summary": summary_rows,
    },
    str(FLAGGED_XLSX),
)
print(f"Saved {len(flagged_df)} flagged / {len(results_df)} judged passages to {FLAGGED_XLSX}")
print(f"Judge audit log (prompts + raw answers): {CACHE_PATH}")
print(f"Citation export: {CITATIONS_XLSX}")

## 5. Reading the results

- **A flag is a review lead.** The judge only sees the excerpt the system stored for a citation, not the whole document. A passage can be flagged because the supporting sentence sits one paragraph outside the excerpt. The card shows exactly what the judge saw, so that case is quick to recognise.
- **`partially_supported` is the interesting bucket.** It is where numbers drift, a district becomes a country, or "may" becomes "does". `unsupported` with high confidence should be rare and is worth checking first.
- **Check the quotes.** A `✗ NOT FOUND` quote means the judge paraphrased or invented evidence; its verdict deserves less trust either way. `≈ near-verbatim` only means the stored source text has PDF extraction artefacts (split or merged words) that the judge silently repaired.
- **Second opinion.** Set `JUDGE_MODEL` to another deployment and re-run section 3; the cache keeps both runs apart, and the Excel can be regenerated per model.
- **Dangling citations** (`[n]` with no stored source) are a data problem in the brief, not a judgement, and are listed under `cannot_assess`.